In [11]:
import time
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold, KFold
import itertools
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    f1_score,
    make_scorer
)

In [12]:
folder = ""

obesity_df_path = folder + "obesity_shuffled_notscaled.csv"
depression_df_path = folder + "depression_shuffled_notscaled.csv"
congressional_df_train_path = folder + "congressional_df_train_preprocessed.csv"
congresional_df_test_path = folder + "congressional_df_test_preprocessed.csv"
rev_df_train_path = folder + "amazon_review_ID.shuf.lrn.csv"
rev_df_test_path = folder + "amazon_review_ID.shuf.tes.csv"


In [13]:
def train_val_split(train_df):
  nrows = train_df.shape[0]

  train_size = int(0.9 * nrows)

  holdout_train_df = train_df[:train_size]
  holdout_val_df = train_df[train_size:]

  return holdout_train_df, holdout_val_df

In [14]:
def train_test_split(full_df):
  nrows = full_df.shape[0]

  train_size = int(0.8 * nrows)

  train_df = full_df[:train_size]
  test_df = full_df[train_size:]

  return train_df, test_df

In [15]:

obesity_df = pd.read_csv(obesity_df_path)
obesity_df_train, obesity_df_test = train_test_split(obesity_df)
depression_df = pd.read_csv(depression_df_path)
depression_df_train, depression_df_test = train_test_split(depression_df)
congressional_df_train = pd.read_csv(congressional_df_train_path)
congressional_df_test = pd.read_csv(congresional_df_test_path)
rev_df_train = pd.read_csv(rev_df_train_path)
if "ID" in rev_df_train.columns:
  rev_df_train = rev_df_train.drop(columns=["ID"])
rev_df_test = pd.read_csv(rev_df_test_path)

In [16]:
obesity_df_train_holdout, obesity_df_val_holdout = train_val_split(obesity_df_train)
depression_df_train_holdout, depression_df_val_holdout = train_val_split(depression_df_train)
congressional_df_train_holdout, congressional_df_val_holdout = train_val_split(congressional_df_train)
rev_df_train_holdout, rev_df_val_holdout = train_val_split(rev_df_train)

In [17]:
def train_deci_tree_with_grid(df, target_attribute, main_scorer):

  scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, average='macro', zero_division=0),
    'recall': make_scorer(recall_score, average='macro', zero_division=0),
    'f1': make_scorer(f1_score, average='macro', zero_division=0),
  }

  param_grid = {
      'criterion': ["entropy", "gini"],
      'min_samples_split': range(2,10,1)
  }

  if df.shape[0] <= 1000:
    param_grid["max_depth"] = range(5,15,1)
  elif df.shape[0] > 1000 and df.shape[0] <= 10000:
    param_grid["max_depth"] = range(10,20,1)
  else:
    param_grid["max_depth"] = range(10,50,1)

  x = df.loc[:, df.columns != target_attribute]
  y_raw = df[target_attribute]
  le = LabelEncoder()
  y = le.fit_transform(y_raw)

  start = time.perf_counter()

  tree = DecisionTreeClassifier(random_state=1)

  cv_strategy = KFold(
    n_splits=5,
    shuffle=True,
    random_state=1
  )

  grid_search = GridSearchCV(
      estimator=tree,
      param_grid=param_grid,
      cv=cv_strategy,
      scoring=scoring,
      refit=main_scorer,
      verbose=True
  )

  grid_search.fit(x,y)

  results = pd.DataFrame(grid_search.cv_results_)
  elapsed = time.perf_counter() - start
  results["Completion_time"] = elapsed
  print("best accuracy", grid_search.best_score_)
  print(grid_search.best_estimator_)
  print("Time(s): ", elapsed)
  return results, le, grid_search.best_estimator_

In [18]:
def train_deci_tree_with_grid_holdout(df_train, df_val, target_attribute, main_scorer):

  scoring = {
    'accuracy': accuracy_score,
    'precision': lambda y_true, y_pred: precision_score(y_true, y_pred, average='macro', zero_division=0),
    'recall': lambda y_true, y_pred: recall_score(y_true, y_pred, average='macro', zero_division=0),
    'f1': lambda y_true, y_pred: f1_score(y_true, y_pred, average='macro', zero_division=0)
  }

  param_grid = {
    'criterion': ["entropy", "gini"],
    'min_samples_split': range(2, 10, 1)
  }

  if df_train.shape[0] <= 1000:
    param_grid["max_depth"] = range(5,15)
  elif df_train.shape[0] <= 10000:
    param_grid["max_depth"] = range(10,20)
  else:
    param_grid["max_depth"] = range(10,50)

  x_train = df_train.loc[:, df_train.columns != target_attribute]
  y_train_raw = df_train[target_attribute]
  le = LabelEncoder()
  le.fit(pd.concat([df_train[target_attribute], df_val[target_attribute]]))
  y_train = le.transform(y_train_raw)

  x_val = df_val.loc[:, df_val.columns != target_attribute]
  y_val_raw = df_val[target_attribute]
  y_val = le.transform(y_val_raw)

  x_full = pd.concat([x_train, x_val], axis=0)
  y_full = np.concatenate([y_train, y_val])

  start = time.perf_counter()

  param_combinations = list(itertools.product(
      param_grid['criterion'],
      param_grid['min_samples_split'],
      param_grid['max_depth']
  ))

  best_score = 0
  best_model = None
  results_list = []

  for criterion, min_samples_split, max_depth in param_combinations:
      model = DecisionTreeClassifier(
          criterion=criterion,
          min_samples_split=min_samples_split,
          max_depth=max_depth,
          random_state=1
      )
      model.fit(x_train, y_train)
      y_pred = model.predict(x_val)

      result = {
          'param_criterion': criterion,
          'param_min_samples_split': min_samples_split,
          'param_max_depth': max_depth,
          'accuracy': accuracy_score(y_val, y_pred),
          'precision': precision_score(y_val, y_pred, average='macro'),
          'recall': recall_score(y_val, y_pred, average='macro'),
          'f1': f1_score(y_val, y_pred, average='macro')
      }

      results_list.append(result)

      if result[main_scorer] > best_score:
          best_score = result[main_scorer]
          best_model = model

  final_model = DecisionTreeClassifier(
      criterion=best_model.get_params()['criterion'],
      min_samples_split=best_model.get_params()['min_samples_split'],
      max_depth=best_model.get_params()['max_depth'],
      random_state=1
  )

  final_model.fit(x_full, y_full)


  elapsed = time.perf_counter() - start
  results = pd.DataFrame(results_list)
  results['Completion_time'] = elapsed

  print("Best", main_scorer, best_score)
  print("Time(s):", elapsed)
  return results, le, final_model


In [19]:
results_obesity_cv, le_obesity_cv, obesity_best_model_cv = train_deci_tree_with_grid(obesity_df_train, "obesity_level_grouped", "accuracy")
results_depression_cv, le_depression_cv, depression_best_model_cv = train_deci_tree_with_grid(depression_df_train, "depression", "accuracy")
results_congressional_cv, le_congressional_cv, congressional_best_model_cv = train_deci_tree_with_grid(congressional_df_train, "class", "f1")
results_rev_cv, le_rev_cv, rev_best_model_cv = train_deci_tree_with_grid(rev_df_train, "Class", "f1")

Fitting 5 folds for each of 160 candidates, totalling 800 fits
best accuracy 0.7778536688146367
DecisionTreeClassifier(criterion='entropy', max_depth=18, random_state=1)
Time(s):  9.686573180370033
Fitting 5 folds for each of 640 candidates, totalling 3200 fits
best accuracy 0.7642629659166676
DecisionTreeClassifier(max_depth=10, min_samples_split=8, random_state=1)
Time(s):  366.84187911543995
Fitting 5 folds for each of 160 candidates, totalling 800 fits
best accuracy 0.9566753819926813
DecisionTreeClassifier(criterion='entropy', max_depth=5, min_samples_split=6,
                       random_state=1)
Time(s):  4.4449948919937015
Fitting 5 folds for each of 160 candidates, totalling 800 fits
best accuracy 0.2083610828055198
DecisionTreeClassifier(max_depth=14, min_samples_split=9, random_state=1)
Time(s):  414.5234458623454


In [20]:
results_obesity_holdout, le_obesity_holdout, obesity_best_model_holdout = train_deci_tree_with_grid_holdout(obesity_df_train_holdout, obesity_df_val_holdout, "obesity_level_grouped", "accuracy")
results_depression_holdout, le_depression_holdout, depression_best_model_holdout = train_deci_tree_with_grid_holdout(depression_df_train_holdout, depression_df_val_holdout, "depression", "accuracy")
results_congressional_holdout, le_congressional_holdout, congressional_best_model_holdout = train_deci_tree_with_grid_holdout(congressional_df_train_holdout, congressional_df_val_holdout, "class", "f1")
results_rev_holdout, le_rev_holdout, rev_best_model_holdout = train_deci_tree_with_grid_holdout(rev_df_train_holdout, rev_df_val_holdout, "Class", "f1")

Best accuracy 0.7869822485207101
Time(s): 1.9786509880796075
Best accuracy 0.7869955156950673
Time(s): 80.3311480274424
Best f1 1.0
Time(s): 0.7903649313375354


/opt/conda/lib/python3.13/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/opt/conda/lib/python3.13/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
/opt/conda/lib/python3.13/site-packages/sklearn/utils/multiclass.py:79: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  ys_types = set(type_of_target(x) for x in ys)
/opt/conda/lib/python3.13/site-packages/sklearn/metrics/_classification.py:98: UserWarning: The number of unique classes is gr

Best f1 0.26365403304178814
Time(s): 91.0960140535608


In [21]:
def get_top_results(results_df, main_scorer):
  mean_score_metrics = ["mean_test_f1", "mean_test_accuracy", "mean_test_precision", "mean_test_recall"]
  if f"mean_test_{main_scorer}" in mean_score_metrics:
    mean_score_metrics.remove(f"mean_test_{main_scorer}")
  mean_score_metrics.insert(0, f"mean_test_{main_scorer}")
  results_df["combined_rank"] = results_df["rank_test_accuracy"] + results_df["rank_test_precision"] + results_df["rank_test_recall"] + results_df["rank_test_f1"]
  results_df_sorted = results_df.sort_values(by=mean_score_metrics, ascending=False)
  param_cols = [col for col in results_df_sorted.columns if 'param_' in col]
  mean_score_metrics.append("combined_rank")
  relevant_cols = mean_score_metrics + param_cols
  results_df_sorted_relevant = results_df_sorted[relevant_cols]

  return results_df_sorted_relevant


In [22]:
def get_top_results_holdout(results_df, main_scorer):
  mean_score_metrics = ["f1", "accuracy", "precision", "recall"]
  if main_scorer in mean_score_metrics:
    mean_score_metrics.remove(main_scorer)
  mean_score_metrics.insert(0, main_scorer)
  results_df_sorted = results_df.sort_values(by=mean_score_metrics, ascending=False)
  param_cols = [col for col in results_df_sorted.columns if 'param_' in col]
  relevant_cols = mean_score_metrics + param_cols
  results_df_sorted_relevant = results_df_sorted[relevant_cols]

  return results_df_sorted_relevant

In [23]:
processed_results_obesity_cv = get_top_results(results_obesity_cv, "accuracy")
processed_results_depression_cv = get_top_results(results_depression_cv, "accuracy")
processed_results_congressional_cv = get_top_results(results_congressional_cv, "f1")
processed_results_rev_cv = get_top_results(results_rev_cv, "f1")

In [24]:
processed_results_obesity_holdout = get_top_results_holdout(results_obesity_holdout, "accuracy")
processed_results_depression_holdout = get_top_results_holdout(results_depression_holdout, "accuracy")
processed_results_congressional_holdout = get_top_results_holdout(results_congressional_holdout, "f1")
processed_results_rev_holdout = get_top_results_holdout(results_rev_holdout, "f1")

In [25]:
processed_results_obesity_cv.head()

,mean_test_accuracy,mean_test_f1,mean_test_precision,mean_test_recall,combined_rank,param_criterion,param_max_depth,param_min_samples_split
64,0.777854,0.739017,0.737992,0.742984,4,entropy,18,2
72,0.777260,0.736750,0.736161,0.739233,8,entropy,19,2
48,0.770753,0.733046,0.730776,0.737795,12,entropy,16,2
56,0.770748,0.732206,0.730639,0.735916,16,entropy,17,2
40,0.770163,0.730136,0.730351,0.731845,21,entropy,15,2


In [26]:
processed_results_obesity_holdout.head()

,accuracy,f1,precision,recall,param_criterion,param_min_samples_split,param_max_depth
65,0.786982,0.753905,0.770941,0.791104,entropy,8,15
66,0.786982,0.753905,0.770941,0.791104,entropy,8,16
67,0.786982,0.753905,0.770941,0.791104,entropy,8,17
68,0.786982,0.753905,0.770941,0.791104,entropy,8,18
69,0.786982,0.753905,0.770941,0.791104,entropy,8,19


In [27]:
processed_results_depression_cv.head()

,mean_test_accuracy,mean_test_f1,mean_test_precision,mean_test_recall,combined_rank,param_criterion,param_max_depth,param_min_samples_split
326,0.764263,0.754391,0.758570,0.751999,4,gini,10,8
327,0.764039,0.754257,0.758291,0.751949,9,gini,10,9
3,0.763680,0.753061,0.758562,0.750305,21,entropy,10,5
320,0.763590,0.753709,0.757828,0.751332,15,gini,10,2
325,0.763456,0.753491,0.757773,0.751066,21,gini,10,7


In [28]:
processed_results_depression_holdout.head()

,accuracy,f1,precision,recall,param_criterion,param_min_samples_split,param_max_depth
160,0.786996,0.779730,0.789880,0.776376,entropy,6,10
0,0.786547,0.779079,0.789804,0.775652,entropy,2,10
240,0.786099,0.778653,0.789245,0.775251,entropy,8,10
40,0.786099,0.778577,0.789408,0.775143,entropy,3,10
120,0.785650,0.778451,0.788217,0.775173,entropy,5,10


In [29]:
processed_results_congressional_cv.head()

,mean_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,combined_rank,param_criterion,param_max_depth,param_min_samples_split
4,0.956675,0.958774,0.95719,0.957777,4,entropy,5,6
12,0.956675,0.958774,0.95719,0.957777,4,entropy,6,6
20,0.956675,0.958774,0.95719,0.957777,4,entropy,7,6
28,0.956675,0.958774,0.95719,0.957777,4,entropy,8,6
36,0.956675,0.958774,0.95719,0.957777,4,entropy,9,6


In [30]:
processed_results_congressional_holdout.head()

,f1,accuracy,precision,recall,param_criterion,param_min_samples_split,param_max_depth
50,1.0,1.0,1.0,1.0,entropy,7,5
51,1.0,1.0,1.0,1.0,entropy,7,6
52,1.0,1.0,1.0,1.0,entropy,7,7
53,1.0,1.0,1.0,1.0,entropy,7,8
54,1.0,1.0,1.0,1.0,entropy,7,9


In [31]:
processed_results_rev_cv.head()

,mean_test_f1,mean_test_accuracy,mean_test_precision,mean_test_recall,combined_rank,param_criterion,param_max_depth,param_min_samples_split
159,0.208361,0.221333,0.224352,0.233247,44,gini,14,9
20,0.207864,0.228000,0.230664,0.225499,30,entropy,7,6
23,0.207819,0.237333,0.233561,0.229361,26,entropy,7,9
153,0.207408,0.220000,0.229436,0.232132,38,gini,14,3
158,0.206498,0.221333,0.220749,0.232812,53,gini,14,8


In [32]:
processed_results_rev_holdout.head()

,f1,accuracy,precision,recall,param_criterion,param_min_samples_split,param_max_depth
54,0.263654,0.306667,0.282313,0.297619,entropy,7,9
24,0.261820,0.306667,0.267730,0.320922,entropy,4,9
53,0.254216,0.306667,0.259375,0.305556,entropy,7,8
43,0.252712,0.293333,0.268924,0.284722,entropy,6,8
65,0.251438,0.266667,0.261111,0.284722,entropy,8,10


In [33]:
def pred_test_data(test_df, model, label_encoder, target_attribute, has_ground_truth):
  x_test = test_df.loc[:, test_df.columns != target_attribute]
  x_test_ids = []
  if "ID" in x_test.columns:
    x_test_ids = x_test["ID"]
    x_test = x_test.drop(columns=["ID"])

  if has_ground_truth:
    y_test_raw = test_df[target_attribute]
    y_test = label_encoder.transform(y_test_raw)


  start = time.perf_counter()

  y_pred = model.predict(x_test)

  elapsed = time.perf_counter() - start
  final_results = pd.DataFrame()
  final_results["time"] = [elapsed]
  final_results["parameters"] = [model.get_params()]
  if has_ground_truth:
    final_results["accuracy"] = [accuracy_score(y_test, y_pred)]
    final_results["precision"] = [precision_score(y_test, y_pred, average="macro")]
    final_results["recall"] = [recall_score(y_test, y_pred, average="macro")]
    final_results["f1"] = [f1_score(y_test, y_pred, average="macro")]
  final_pred = x_test.copy()
  final_pred["y_pred"] = label_encoder.inverse_transform(y_pred)
  if len(x_test_ids) > 0:
    final_pred["id"] = x_test_ids

  return final_results, final_pred


In [34]:
prediction_results_obesity_cv, y_pred_obesity_cv = pred_test_data(obesity_df_test, obesity_best_model_cv, le_obesity_cv, "obesity_level_grouped", True)
prediction_results_depression_cv, y_pred_depression_cv = pred_test_data(depression_df_test, depression_best_model_cv, le_depression_cv, "depression", True)
prediction_results_congressional_cv, y_pred_congressional_cv = pred_test_data(congressional_df_test, congressional_best_model_cv, le_congressional_cv, "class", False)
prediction_results_rev_cv, y_pred_rev_cv = pred_test_data(rev_df_test, rev_best_model_cv, le_rev_cv, "Class", False)

In [35]:
prediction_results_obesity_holdout, y_pred_obesity_holdout = pred_test_data(obesity_df_test, obesity_best_model_holdout, le_obesity_holdout, "obesity_level_grouped", True)
prediction_results_depression_holdout, y_pred_depression_holdout = pred_test_data(depression_df_test, depression_best_model_holdout, le_depression_holdout, "depression", True)
prediction_results_congressional_holdout, y_pred_congressional_holdout = pred_test_data(congressional_df_test, congressional_best_model_holdout, le_congressional_holdout, "class", False)
prediction_results_rev_holdout, y_pred_rev_holdout = pred_test_data(rev_df_test, rev_best_model_holdout, le_rev_holdout, "Class", False)

In [36]:
prediction_results_obesity_cv.head()

,time,parameters,accuracy,precision,recall,f1
0,0.000962,"{'ccp_alpha': 0.0, 'class_weight': None, 'crit...",0.794326,0.713317,0.710919,0.706864


In [37]:
prediction_results_obesity_holdout.head()

,time,parameters,accuracy,precision,recall,f1
0,0.000877,"{'ccp_alpha': 0.0, 'class_weight': None, 'crit...",0.801418,0.729359,0.73949,0.73256


In [38]:
prediction_results_depression_cv.head()

,time,parameters,accuracy,precision,recall,f1
0,0.001946,"{'ccp_alpha': 0.0, 'class_weight': None, 'crit...",0.770224,0.763402,0.758769,0.760704


In [39]:
prediction_results_congressional_cv.head()

,time,parameters
0,0.000595,"{'ccp_alpha': 0.0, 'class_weight': None, 'crit..."


In [40]:
prediction_results_congressional_holdout.head()

,time,parameters
0,0.000636,"{'ccp_alpha': 0.0, 'class_weight': None, 'crit..."


In [41]:
prediction_results_rev_cv.head()

,time,parameters
0,0.027411,"{'ccp_alpha': 0.0, 'class_weight': None, 'crit..."


In [42]:
prediction_results_rev_holdout.head()

,time,parameters
0,0.027155,"{'ccp_alpha': 0.0, 'class_weight': None, 'crit..."


In [43]:
def kaggle_comp_file(pred_df):
  pred_df_final = pred_df[["id", "y_pred"]].rename(columns={"y_pred": "class", "id" : "ID"})
  return pred_df_final


In [44]:
kaggle_submission_congressional_cv = kaggle_comp_file(y_pred_congressional_cv)
kaggle_submission_congressional_holdout = kaggle_comp_file(y_pred_congressional_holdout)

kaggle_submission_congressional_cv.to_csv('congressional_decision_tree_cv_submission_group39.csv', index=False)
kaggle_submission_congressional_holdout.to_csv('congressional_decision_tree_holdout_submission_group39.csv', index=False)

In [45]:
kaggle_submission_rev_cv = kaggle_comp_file(y_pred_rev_cv)
kaggle_submission_rev_holdout = kaggle_comp_file(y_pred_rev_holdout)

kaggle_submission_rev_cv.to_csv('reviews_decision_tree_cv_submission_group39.csv', index=False)
kaggle_submission_rev_holdout.to_csv('reviews_decision_tree_holdout_submission_group39.csv', index=False)

In [46]:
y_pred_congressional_cv.head()

,handicapped-infants,water-project-cost-sharing,adoption-of-the-budget-resolution,physician-fee-freeze,el-salvador-aid,religious-groups-in-schools,anti-satellite-test-ban,aid-to-nicaraguan-contras,mx-missile,immigration,synfuels-crporation-cutback,education-spending,superfund-right-to-sue,crime,duty-free-exports,export-administration-act-south-africa,y_pred,id
0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,democrat,190
1,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.5,1.0,democrat,285
2,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,republican,251
3,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,democrat,40
4,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,democrat,91
